# !! Title : Customer Purchase Behavior Analysis !!

In [1]:
import pandas as pd 
import numpy as np 
import json 
import matplotlib.pyplot as plt
import seaborn as sns
# !pip install openpyxl

# STEP 1: DATA LOADING

### 1a. Load users from Excel (XLSX)

In [2]:
users_df = pd.read_excel("users.xlsx", sheet_name="users.csv")
print(f"users.xlsx loaded  --> {users_df.shape[0]} rows, {users_df.shape[1]} cols")
users_df.head(5)

users.xlsx loaded  --> 200 rows, 6 cols


,user_id,name,age,gender,city,registration_date
0,U0001,Vihaan Sharma,35,Other,Jaipur,2022-09-08
1,U0002,Sai Reddy,30,Other,Hyderabad,2023-11-24
2,U0003,Aarohi Gupta,37,Other,Indore,2022-02-02
3,U0004,Aarav Gupta,44,Male,Kolkata,2023-06-02
4,U0005,Sara Sharma,30,Other,Chennai,2024-01-04


### # 1b. Load sales from JSON

In [3]:
sales_df= pd.read_json("sales.json")
print(f"sales.json loaded  --> {sales_df.shape[0]} rows, {sales_df.shape[1]} cols")
print(sales_df.head())

sales.json loaded  --> 1000 rows, 6 cols
  transaction_id user_id product_id  amount payment_type       date
0        T000001   U0024       P015   67.67       Wallet 2023-02-12
1        T000002   U0196       P044   76.44          UPI 2023-03-24
2        T000003   U0196       P049  104.57   Debit Card 2025-08-21
3        T000004   U0133       P042  102.75  Net Banking 2024-07-23
4        T000005   U0047       P038   23.89  Net Banking 2025-10-04


### 1c. Load inventory from SQL

In [4]:
import sqlite3
conn = sqlite3.connect(':memory:')

In [5]:
with sqlite3.connect("inventory.db") as conn:
    with open('inventory.sql') as f:
        conn.executescript(f.read())

    inventory_df = pd.read_sql('SELECT * FROM products', conn)

print(f"inventory.sql loaded --> {inventory_df.shape[0]} rows, {inventory_df.shape[1]} cols")
inventory_df.head(5)

inventory.sql loaded --> 50 rows, 5 cols


,product_id,product_name,category,price,stock
0,P001,Product_001,Grocery,264.89,371
1,P002,Product_002,Grocery,605.91,150
2,P003,Product_003,Beauty,3027.98,127
3,P004,Product_004,Toys,2600.12,229
4,P005,Product_005,Books,1178.99,18


### Data info

In [6]:
print("\n --- Data Types & Missing Values ---")
for name, df in [("User",users_df), ("Sales",sales_df), ("Inventory" ,inventory_df)] : 
    print(f"\n {name}")
    print(df.dtypes.to_string())
    print(f"-> Missing Values : {df.isnull().sum().sum()}")
    


 --- Data Types & Missing Values ---

 User
user_id                      object
name                         object
age                           int64
gender                       object
city                         object
registration_date    datetime64[ns]
-> Missing Values : 0

 Sales
transaction_id            object
user_id                   object
product_id                object
amount                   float64
payment_type              object
date              datetime64[ns]
-> Missing Values : 0

 Inventory
product_id       object
product_name     object
category         object
price           float64
stock             int64
-> Missing Values : 0


In [7]:
orig_users    = len(users_df)
orig_sales    = len(sales_df)
orig_inv      = len(inventory_df)
mv_before_all = (users_df.isnull().sum().sum()
                 + sales_df.isnull().sum().sum()
                 + inventory_df.isnull().sum().sum())

# STEP 2: DATA CLEANING

### 1. Impute missing numeric: mean strategy

In [8]:
from sklearn.impute import SimpleImputer , KNNImputer
num_imputer = SimpleImputer(strategy='mean')
users_df['age'] = num_imputer.fit_transform(users_df[['age']])

print("Mean used for filling:", num_imputer.statistics_[0])

Mean used for filling: 31.26


### 2.Impute missing categorical: most-frequent strategy


In [9]:
cat_imputer = SimpleImputer(strategy='most_frequent')
for col in ['gender', 'city', 'name']:
    if col in users_df.columns and users_df[col].isnull().any():
        users_df[[col]] = cat_imputer.fit_transform(users_df[[col]])
        print("Most Frequent Used:", cat_imputer.statistics_[0])

### 3. KNN Imputer on numerical columns

In [10]:
num_cols_users = users_df.select_dtypes(include=np.number).columns.tolist()
if num_cols_users:
    knn_imp = KNNImputer(n_neighbors=5)
    users_df[num_cols_users] = knn_imp.fit_transform(users_df[num_cols_users])

mv_after_all = (users_df.isnull().sum().sum()
                + sales_df.isnull().sum().sum()
                + inventory_df.isnull().sum().sum())
print(f"  Missing values → Before: {mv_before_all}  |  After: {mv_after_all}")


  Missing values → Before: 0  |  After: 0


# STEP 3: OUTLIER HANDLING


In [11]:
outliers_before = len(sales_df)
outliers_before

1000

### 1. Z-score Method

In [12]:
from scipy import stats

amount_clean = sales_df["amount"].dropna()
z_scores = np.abs(stats.zscore(amount_clean))
removed_z = int((z_scores >= 3).sum())
print(f"Z-score method removes : {removed_z} rows")

Z-score method removes : 15 rows


### 2. IQR Method

In [13]:
Q1 = sales_df['amount'].quantile(0.25) 
Q3 = sales_df['amount'].quantile(0.75)
IQR = Q3 - Q1
mask_iqr = (sales_df['amount'] >= Q1 - 1.5*IQR) & (sales_df['amount'] <= Q3 + 1.5*IQR)
removed_iqr = int((~mask_iqr).sum())
print(f"  IQR method would remove     : {removed_iqr} rows")


  IQR method would remove     : 53 rows


In [14]:
sales_df = sales_df[mask_iqr].reset_index(drop=True)
outliers_removed = outliers_before - len(sales_df)


In [15]:
from scipy.stats.mstats import winsorize
inventory_df['price_winsorized'] = winsorize(inventory_df['price'], limits=[0.05, 0.05])
print(f"  Winsorization applied to inventory price")

  Winsorization applied to inventory price


In [16]:
print(f"  Outliers removed from sales : {outliers_removed}")

  Outliers removed from sales : 53


# STEP 4: DATA TRANSFORMATION

In [17]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer

In [18]:
sales_df['year']  = sales_df['date'].dt.year
sales_df['month'] = sales_df['date'].dt.month
sales_df['day']   = sales_df['date'].dt.day
print("  Date split into year / month / day") 

  Date split into year / month / day


### --- Label Encoding for binary Columns ---

In [19]:
le = LabelEncoder()
users_df['gender_encoded'] = le.fit_transform(users_df['gender'].astype(str))
print("Label encoding -> gender.")

Label encoding -> gender.


In [20]:
print(users_df.columns)

Index(['user_id', 'name', 'age', 'gender', 'city', 'registration_date',
       'gender_encoded'],
      dtype='object')


### --- One hot Encoding for nominal columns ---

In [22]:
ohe = OneHotEncoder(drop='first', sparse_output=False)

In [25]:
# users_df -> city
city_encoded = ohe.fit_transform(users_df[['city']])
city_cols = ohe.get_feature_names_out(['city'])
users_df[city_cols] = city_encoded
users_df[city_cols].head(3)

,city_Bengaluru,city_Bhopal,city_Chennai,city_Delhi,city_Ghaziabad,city_Hyderabad,city_Indore,city_Jaipur,city_Kanpur,city_Kolkata,city_Lucknow,city_Mumbai,city_Nagpur,city_Patna,city_Pune,city_Surat,city_Thane,city_Vadodara,city_Visakhapatnam
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
# sales_df -> payment_type
pay_encoded = ohe.fit_transform(sales_df[['payment_type']])
pay_cols = ohe.get_feature_names_out(['payment_type'])
sales_df[pay_cols] = pay_encoded
sales_df[pay_cols].head(3)

,payment_type_Credit Card,payment_type_Debit Card,payment_type_Net Banking,payment_type_UPI,payment_type_Wallet
0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0,0.0


In [30]:
# inventory_df -> category
cat_encoded = ohe.fit_transform(inventory_df[['category']])
cat_cols = ohe.get_feature_names_out(['category'])
inventory_df[cat_cols] = cat_encoded 
inventory_df[cat_cols].head(5)

,category_Books,category_Clothing,category_Electronics,category_Grocery,category_Home,category_Sports,category_Toys
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0
